# 🛡️ PrédiSinistre — Détection de Fraude Assurantielle
## Rendu 1 : Exploration, Data Visualisation & Pre-processing

---

| Champ              | Détail                                              |
|--------------------|-----------------------------------------------------|
| **Cursus**         | Data Scientist — DataScientest                      |
| **Difficulté**     | 6 / 10                                              |
| **Dataset**        | `insurance_claims.csv` (1 000 lignes × 40 colonnes) |
| **Variable cible** | `fraud_reported` (Y = fraude / N = non-fraude)      |
| **Objectif**       | Détecter automatiquement les sinistres frauduleux   |

---

## 📋 Plan du notebook

1. [Imports & Configuration](#1)
2. [Chargement et première inspection](#2)
3. [Analyse univariée](#3)
4. [Visualisation — 6 graphiques commentés](#4)
5. [Nettoyage et préparation du dataset](#5)
6. [Encodage et Feature Engineering](#6)
7. [Export du dataset propre](#7)
8. [Modèles de base (Baseline)](#8)
9. [Interprétation et perspectives](#9)

---
## 1. Imports & Configuration <a id='1'></a>

On importe toutes les bibliothèques nécessaires en une seule cellule, conformément aux bonnes pratiques :
- **pandas / numpy** : manipulation et calcul sur données tabulaires
- **matplotlib / seaborn** : visualisation
- **sklearn** : prétraitement, modèles et évaluation

On fixe également les graines aléatoires (`RANDOM_STATE`) pour garantir la **reproductibilité** des résultats.

In [1]:
# ─────────────────────────────────────────────────────────────────
# BLOC 1 — IMPORTS & CONFIGURATION GLOBALE
# ─────────────────────────────────────────────────────────────────

# Manipulation de données
import pandas as pd
import numpy as np

# Visualisation
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from matplotlib.gridspec import GridSpec

# Prétraitement & sélection de features
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline

# Modèles de Machine Learning
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier  # modèle naïf de référence

# Métriques d'évaluation
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    f1_score
)

# Utilitaires
import warnings
warnings.filterwarnings('ignore')  # masquer les avertissements non critiques

# ── Constantes de configuration ──────────────────────────────────
RANDOM_STATE = 42          # graine pour la reproductibilité
TEST_SIZE    = 0.20        # proportion du jeu de test
DATA_PATH    = 'insurance_claims.csv'  # chemin vers le dataset brut
OUTPUT_PATH  = 'insurance_claims_clean.csv'  # chemin de sortie

# ── Palette de couleurs cohérente dans tout le notebook ──────────
COLOR_NON_FRAUD = '#2E86AB'   # bleu  → sinistres légitimes
COLOR_FRAUD     = '#E84855'   # rouge → sinistres frauduleux
PALETTE_GENERAL = ['#2E86AB', '#E84855', '#3BB273', '#F9C74F', '#9B5DE5']

# ── Style global des graphiques ──────────────────────────────────
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.dpi'      : 120,
    'font.family'     : 'DejaVu Sans',
    'axes.titlesize'  : 13,
    'axes.labelsize'  : 11,
    'legend.fontsize' : 10,
    'xtick.labelsize' : 9,
    'ytick.labelsize' : 9,
})

print('✅ Imports et configuration terminés.')

✅ Imports et configuration terminés.


---
## 2. Chargement et première inspection du dataset <a id='2'></a>

Cette étape vise à répondre à trois questions fondamentales :
1. **Quelle est la structure du dataset ?** (dimensions, types de colonnes)
2. **Y a-t-il des données manquantes ?** (NaN natifs, valeurs encodées '?')
3. **Quelle est la distribution de la variable cible ?** (déséquilibre de classes)

> 📌 **Règle méthodologique** : on n'effectue AUCUNE transformation à ce stade — on observe uniquement.

In [4]:
# ─────────────────────────────────────────────────────────────────
# BLOC 2.1 — CHARGEMENT DU DATASET
# ─────────────────────────────────────────────────────────────────

df = pd.read_csv("../data/raw/insurance_claims.csv")

# Affichage des dimensions et des premières lignes du dataset
print('=' * 60)
print(f'  Dimensions : {df.shape[0]} lignes × {df.shape[1]} colonnes')
print('=' * 60)

df.head()

  Dimensions : 1000 lignes × 40 colonnes


,months_as_customer,age,policy_number,policy_bind_date,policy_state,policy_csl,policy_deductable,policy_annual_premium,umbrella_limit,insured_zip,...,police_report_available,total_claim_amount,injury_claim,property_claim,vehicle_claim,auto_make,auto_model,auto_year,fraud_reported,_c39
0,328,48,521585,2014-10-17,OH,250/500,1000,1406.91,0,466132,...,YES,71610,6510,13020,52080,Saab,92x,2004,Y,NaN
1,228,42,342868,2006-06-27,IN,250/500,2000,1197.22,5000000,468176,...,?,5070,780,780,3510,Mercedes,E400,2007,Y,NaN
2,134,29,687698,2000-09-06,OH,100/300,2000,1413.14,5000000,430632,...,NO,34650,7700,3850,23100,Dodge,RAM,2007,N,NaN
3,256,41,227811,1990-05-25,IL,250/500,2000,1415.74,6000000,608117,...,NO,63400,6340,6340,50720,Chevrolet,Tahoe,2014,Y,NaN
4,228,44,367455,2014-06-06,IL,500/1000,1000,1583.91,6000000,610706,...,NO,6500,1300,650,4550,Accura,RSX,2009,N,NaN


In [ ]:
# ─────────────────────────────────────────────────────────────────
# BLOC 2.2 — TYPES ET VALEURS MANQUANTES
# ─────────────────────────────────────────────────────────────────

# Résumé des types de données : distinguer variables numériques / catégorielles
dtype_summary = pd.DataFrame({
    'Type'           : df.dtypes,
    'Valeurs_nulles' : df.isnull().sum(),
    '% Nulles'       : (df.isnull().sum() / len(df) * 100).round(2),
    'Nb_uniques'     : df.nunique(),
})

# Détection des colonnes contenant des '?' (valeurs manquantes masquées)
question_mark_counts = {}
for col in df.select_dtypes(include=['object', 'str']).columns:
    count = (df[col] == '?').sum()
    if count > 0:
        question_mark_counts[col] = count


dtype_summary['Nb_points_interrogation'] = 0
for col, count in question_mark_counts.items():
    dtype_summary.loc[col, 'Nb_points_interrogation'] = count

# Affichage du résumé des types et des valeurs manquantes
print('─── Résumé des colonnes ───────────────────────────────────')
print(dtype_summary.to_string())
print()
print(f'⚠️  Colonnes avec des points d\'interrogation : {list(question_mark_counts.keys())}')

─── Résumé des colonnes ───────────────────────────────────
                                Type  Valeurs_nulles  % Nulles  Nb_uniques  Nb_points_interrogation
months_as_customer             int64               0       0.0         391                        0
age                            int64               0       0.0          46                        0
policy_number                  int64               0       0.0        1000                        0
policy_bind_date                 str               0       0.0         951                        0
policy_state                     str               0       0.0           3                        0
policy_csl                       str               0       0.0           3                        0
policy_deductable              int64               0       0.0           3                        0
policy_annual_premium        float64               0       0.0         991                        0
umbrella_limit                 int64    